In [1]:
!pip install -q transformers datasets torch nltk streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 34.2 MB/s eta 0:00:00


In [2]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [5]:
!wget https://sumith1896.github.io/spoc/data/spoc.zip
!unzip spoc.zip


--2025-10-31 14:35:53--  https://sumith1896.github.io/spoc/data/spoc.zip
Resolving sumith1896.github.io (sumith1896.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to sumith1896.github.io (sumith1896.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9960355 (9.5M) [application/x-zip-compressed]
Saving to: ‘spoc.zip’

spoc.zip            100%[===================>]   9.50M  --.-KB/s    in 0.03s   

2025-10-31 14:35:54 (314 MB/s) - ‘spoc.zip’ saved [9960355/9960355]

Archive:  spoc.zip
   creating: test/
  inflating: test/spoc-testw.tsv     
  inflating: test/spoc-testp.tsv     
   creating: testcases/
   creating: testcases/1000A/
  inflating: testcases/1000A/1000A_testcases.txt  
  inflating: testcases/1000A/1000A_testcases_hidden.txt  
  inflating: testcases/1000A/1000A_testcases_public.txt  
   creating: testcases/1003A/
  inflating: testcases/1003A/1003A_testcases.txt  
  inflating: testcases/1003A/100

In [67]:
import pandas as pd
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from nltk.translate.bleu_score import sentence_bleu

In [68]:
train_df = pd.read_csv("train/split/spoc-train-train.tsv", sep='\t')
val_df   = pd.read_csv("train/split/spoc-train-eval.tsv", sep='\t')
test_df  = pd.read_csv("train/split/spoc-train-test.tsv", sep='\t')


In [69]:
print(train_df.columns)


Index(['text', 'code', 'workerid', 'probid', 'subid', 'line', 'indent'], dtype='object')


In [70]:

train_df = train_df[['text', 'code']]
val_df   = val_df[['text', 'code']]
test_df  = test_df[['text', 'code']]


train_df.columns = ['pseudo_code', 'code']
val_df.columns   = ['pseudo_code', 'code']
test_df.columns  = ['pseudo_code', 'code']


for df in [train_df, val_df, test_df]:
    df['pseudo_code'] = df['pseudo_code'].str.strip()
    df['code'] = df['code'].str.strip()

print("Train samples:", len(train_df))
train_df.head()


Train samples: 246086


,pseudo_code,code
0,NaN,int main() {
1,create string s,string s;
2,"create integers x1, y1, x2, y2","int x1, y1, x2, y2;"
3,read s,cin >> s;
4,set x1 to s[0] - 96,x1 = s[0] - 96;


In [71]:
train_df_small = train_df.sample(min(5000, len(train_df)), random_state=42)
val_df_small   = val_df.sample(min(1000, len(val_df)), random_state=42)


In [72]:
def format_example(pseudo, code):
    return f"<pseudo> {pseudo} <code> {code}"

train_texts = [format_example(p, c) for p, c in zip(train_df_small['pseudo_code'], train_df_small['code'])]
val_texts   = [format_example(p, c) for p, c in zip(val_df_small['pseudo_code'], val_df_small['code'])]

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256, return_tensors="pt")
val_encodings   = tokenizer(val_texts, truncation=True, padding=True, max_length=256, return_tensors="pt")

In [73]:
class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = item['input_ids']
        return item

train_dataset = CodeDataset(train_encodings)
val_dataset   = CodeDataset(val_encodings)

In [53]:
model = GPT2LMHeadModel.from_pretrained("distilgpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [54]:
training_args = TrainingArguments(
    output_dir="./distilgpt2_spoc",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_dir='./logs',
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    learning_rate=5e-5,
    report_to=[]
)


In [55]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [56]:
trainer.train()

Step,Training Loss
100,0.594100
200,0.313200
300,0.291100
400,0.301800
500,0.265100
600,0.266500
700,0.238100
800,0.253500
900,0.256200
1000,0.205400


TrainOutput(global_step=7500, training_loss=0.19460065981547037, metrics={'train_runtime': 1221.3738, 'train_samples_per_second': 12.281, 'train_steps_per_second': 6.141, 'total_flos': 428689981440000.0, 'train_loss': 0.19460065981547037, 'epoch': 3.0})

In [74]:

trainer.save_model("./final_gpt2")        # saves model weights and config
tokenizer.save_pretrained("./final_gpt2") # saves tokenizer files

import os
os.listdir("./final_gpt2")


['model.safetensors',
 'vocab.json',
 'generation_config.json',
 'tokenizer_config.json',
 'training_args.bin',
 'merges.txt',
 'config.json',
 'special_tokens_map.json']

In [75]:
def evaluate_bleu(model, tokenizer, pseudo, reference_code):
    model.eval()
    input_text = f"<pseudo> {pseudo} <code>"
    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_length=100)
    generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True).split("<code>")[-1].strip()
    score = sentence_bleu([reference_code.split()], generated_code.split())
    return generated_code, score


In [78]:
pseudo_example = "create integers x1, y1, x2, y2	"

# Generate code from model
def generate_code(model, tokenizer, pseudo_example, max_length=50):
    model.eval()


    prompt = f"### Pseudo-code:\n{pseudo_example}\n### C++ code:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=True,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode and take only the part after "### C++ code:"
    gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    code = gen_text.split("### C++ code:")[-1].strip()

    return code

gen_code = generate_code(model, tokenizer, pseudo_example)
print("Generated Code:\n", gen_code)


Generated Code:
 int x1, x2, y1, x2, y2;


In [79]:
import gradio as gr
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Function to generate C++ code
def generate_cpp_code(pseudo_example, max_length=100):
    model.eval()
    prompt = f"### Pseudo-code:\n{pseudo_example}\n### C++ code:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=True,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Keep only the code after the C++ code separator
    code = gen_text.split("### C++ code:")[-1].strip()
    return code

# Gradio Interface
iface = gr.Interface(
    fn=generate_cpp_code,
    inputs=gr.Textbox(lines=5, placeholder="Enter pseudo-code here..."),
    outputs=gr.Textbox(label="Generated C++ Code"),
    title="Pseudo-code → C++ Code Generator",
    description="Enter your pseudo-code and the model will generate corresponding C++ code."
)

# Launch interface
iface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f595856b3ae3842605.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
